# 2. Advanced Threat Hunting

## Hypothesis-driven hunting

Professional threat hunting follows a cycle:

1. **Hypothesis**: "I think an attacker used compromised credentials to access our database server."
2. **Query**: Write KQL/API queries to find evidence.
3. **Analyze**: Do the results support or refute the hypothesis?
4. **Action**: If confirmed → create detection rule + incident. If not → refine hypothesis.

### Bad → better → best hunting

| Level | Approach |
|-------|----------|
| 🔴 Bad | One-off hunt: find attacker, write a ticket, move on. Same attack tomorrow = same manual work. |
| 🟡 Better | Hunt with entity pivoting: follow the compromised user/IP/host across tables to build the full story. |
| 🟢 Best | Enrich with **threat intel / watchlists**, then **convert successful hunts into scheduled detection rules + playbooks** so they're caught automatically next time. |

We'll practice all three levels with four hunting hypotheses.


In [ ]:
import httpx, json
from collections import Counter, defaultdict

SIEM = 'http://localhost:8000'

def query(table, filter=None, aggregate_by=None, limit=500):
    r = httpx.post(f'{SIEM}/query', json={'table_name': table, 'filter': filter, 'aggregate_by': aggregate_by, 'limit': limit})
    return r.json()['results']

# ===== HYPOTHESIS 1: Account compromise via brute force =====
# MITRE ATT&CK: T1110 Brute Force → T1078 Valid Accounts
#               TA0006 Credential Access → TA0001 Initial Access
print('╔══════════════════════════════════════════════════════════════╗')
print('║ H1  An account was compromised via brute force              ║')
print('║     ATT&CK: T1110 Brute Force + T1078 Valid Accounts        ║')
print('╚══════════════════════════════════════════════════════════════╝\n')

all_signins = query('SigninLogs', limit=1000)
user_signin = defaultdict(lambda: {'fail': 0, 'success': 0, 'fail_ips': set(), 'success_ips': set(), 'locs': set()})
for s in all_signins:
    u = s['UserPrincipalName']
    if s['ResultType'] == 'Failure':
        user_signin[u]['fail'] += 1
        user_signin[u]['fail_ips'].add(s['IPAddress'])
    else:
        user_signin[u]['success'] += 1
        user_signin[u]['success_ips'].add(s['IPAddress'])
    user_signin[u]['locs'].add(s['Location'])

compromised = [(u, d) for u, d in user_signin.items() if d['fail'] > 5 and d['success'] > 0]
if compromised:
    print('✅ HYPOTHESIS CONFIRMED\n')
    for user, data in compromised:
        print(f'  Account: {user}')
        print(f'  Failed attempts: {data["fail"]} from IPs: {sorted(data["fail_ips"])}')
        print(f'  Successful logins: {data["success"]} from IPs: {sorted(data["success_ips"])}')
        print(f'  Locations: {sorted(data["locs"])}')
        overlap = data['fail_ips'] & data['success_ips']
        if overlap:
            print(f'  ⚠️  Same IP used for failures AND success: {sorted(overlap)}')
            print(f'  → Attacker succeeded after brute forcing!')
else:
    print('❌ HYPOTHESIS NOT CONFIRMED — no brute force patterns found.')


In [ ]:
# ===== HYPOTHESIS 2: Compromised account moved laterally =====
# MITRE ATT&CK: T1021 Remote Services (TA0008 Lateral Movement),
#               T1003 OS Credential Dumping (TA0006 Credential Access)
print('╔══════════════════════════════════════════════════════════════╗')
print('║ H2  Compromised account moved laterally to servers          ║')
print('║     ATT&CK: T1021 Remote Services + T1003 Credential Dumping║')
print('╚══════════════════════════════════════════════════════════════╝\n')

if compromised:
    target_user = compromised[0][0].split('@')[0]  # e.g. 'alice'

    endpoint_events = query('DeviceEvents', filter={'AccountName': target_user})

    devices   = Counter(e['DeviceName'] for e in endpoint_events)
    processes = Counter(e['FileName']   for e in endpoint_events)
    remote_exec  = [e for e in endpoint_events if e['ActionType'] == 'RemoteExecution']
    attack_tool_names = {'mimikatz.exe', 'psexec.exe', 'certutil.exe', 'nc.exe'}
    attack_tools_seen = sorted({e['FileName'] for e in endpoint_events if e['FileName'] in attack_tool_names})

    print(f'User: {target_user}')
    print(f'Devices touched ({len(devices)}): {dict(devices)}')
    print(f'Processes run:   {dict(processes)}')
    print(f'Remote executions: {len(remote_exec)}')
    print(f'Attack tools: {attack_tools_seen}')

    if len(devices) > 1 and remote_exec and attack_tools_seen:
        print(f'\n✅ HYPOTHESIS CONFIRMED')
        print(f'  {target_user} accessed {len(devices)} devices with remote execution.')
        print(f'  Attack tools used: {", ".join(attack_tools_seen)}')
        print(f'\n  → ACTION: Isolate all affected devices, disable user, investigate data access.')
    else:
        print('\n❌ HYPOTHESIS NOT CONFIRMED')


In [ ]:
# ===== HYPOTHESIS 3: Data exfiltrated to external destinations =====
# MITRE ATT&CK: T1041 Exfiltration Over C2 Channel (TA0010 Exfiltration)
print('╔═══════════════════════════════════════════════════════════╗')
print('║ H3  Data was exfiltrated to external endpoints           ║')
print('║     ATT&CK: T1041 Exfiltration Over C2 Channel           ║')
print('╚═══════════════════════════════════════════════════════════╝\n')

fw_events = query('AzureFirewall', limit=500)
external = [e for e in fw_events if not e.get('DestinationIP', '').startswith('10.')]
external_ips = Counter(e['DestinationIP'] for e in external)

known_bad_ips = {'185.220.101.42', '45.33.32.156', '198.51.100.99'}

print(f'Total firewall events: {len(fw_events)}')
print(f'External connections: {len(external)}\n')
print('External destination IPs:')

exfil_confirmed = False
for ip, count in external_ips.most_common():
    threat = '🔴 KNOWN C2/EXFIL' if ip in known_bad_ips else '⬜'
    if ip in known_bad_ips:
        exfil_confirmed = True
    print(f'  {ip:<20} {count:>3} connections  {threat}')

if exfil_confirmed:
    # Entity pivot: trace which internal host connected to bad IPs
    for bad_ip in known_bad_ips:
        sources = [e['SourceIP'] for e in external if e['DestinationIP'] == bad_ip]
        if sources:
            print(f'\n  {bad_ip} was contacted by: {Counter(sources).most_common()}')

    # Cross-reference with endpoint events
    upload_events = query('DeviceEvents', filter={'ActionType': 'FileUploaded'})
    if upload_events:
        print(f'\n  File upload events: {len(upload_events)}')
        devices = Counter(e['DeviceName'] for e in upload_events)
        print(f'  Uploading devices: {dict(devices)}')

    print(f'\n✅ HYPOTHESIS CONFIRMED — data exfiltration detected!')
    print(f'  → ACTION: Block IPs in firewall, isolate source hosts, assess data loss.')
else:
    print('\n❌ HYPOTHESIS NOT CONFIRMED')


## Hypothesis 4 — Threat-intel enrichment with watchlists

**Watchlists** are lookup tables of known-bad (or known-good) indicators — IPs, domains, users, file hashes — typically fed by your threat-intel team. In Sentinel they appear as the `_GetWatchlist("name")` function used inside KQL.

Instead of hard-coding IOCs inside every hunt (bad), store them once in a watchlist and match any table against them (best).


In [ ]:
# ===== HYPOTHESIS 4: Any activity matches our threat intel IOCs? =====
# MITRE ATT&CK: cross-cutting (TI enrichment supports all tactics)

# 1) Create / update the watchlist of known-bad IPs
bad_ip_list = {
    'name': 'TI_BadIPs',
    'description': 'Known C2 / anonymizer / exfiltration IPs from threat intel feed',
    'items': ['185.220.101.42', '45.33.32.156', '198.51.100.99'],
}
r = httpx.post(f'{SIEM}/watchlists', json=bad_ip_list)
print(f'Watchlist: {r.json()}')

# 2) Match the watchlist against multiple tables
print('\n╔═══════════════════════════════════════════════════════════╗')
print('║ H4  Any SIEM activity matches known-bad IP watchlist?    ║')
print('╚═══════════════════════════════════════════════════════════╝\n')

for table, field in [('SigninLogs', 'IPAddress'), ('AzureFirewall', 'DestinationIP')]:
    r = httpx.post(f'{SIEM}/watchlists/match', json={
        'watchlist': 'TI_BadIPs',
        'table_name': table,
        'field': field,
        'time_range_minutes': 1440,
        'limit': 50,
    })
    data = r.json()
    hits = data.get('results') or data.get('matches') or []
    print(f'  {table}.{field}: {len(hits)} hit(s)')
    for h in hits[:3]:
        # show a compact preview
        preview = {k: h.get(k) for k in (field, 'UserPrincipalName', 'SourceIP') if h.get(k)}
        print(f'    • {preview}')
    if len(hits) > 3:
        print(f'    … {len(hits)-3} more')

print('\nReal-KQL equivalent:')
print('  let BadIPs = _GetWatchlist("TI_BadIPs") | project IP = SearchKey;')
print('  SigninLogs | where IPAddress in (BadIPs)')
print('  | union (AzureFirewall | where DestinationIP in (BadIPs))')


## From hunt to detection rule — the "best" level

When your hunt finds something real, **convert it to an analytics rule** so it's detected automatically next time. This is the difference between a hunting tool and a SOC capability.


In [ ]:
# Convert our exfiltration hunt into a permanent detection rule
print('=== Converting hunt to detection rule ===\n')

new_rule = {
    'name': 'High-volume outbound to external IP',
    'severity': 'High',
    'tactic': 'Exfiltration',
    'query_table': 'AzureFirewall',
    'aggregate_by': 'DestinationIP',
    'threshold': 5,
    'window_minutes': 60,
    'description': 'Detects >5 outbound connections to a single external IP in 1 hour. May indicate data exfiltration.',
}

r = httpx.post(f'{SIEM}/rules', json=new_rule)
print(f'Rule created: {r.json()}')
print(f'\nReal KQL equivalent:')
print('''  AzureFirewall
  | where TimeGenerated > ago(1h)
  | where DestinationIP !startswith "10."
  | summarize ConnectionCount=count() by DestinationIP, SourceIP
  | where ConnectionCount > 5''')

print(f'\n💡 This is the threat hunting cycle:')
print(f'   Hypothesis → Query → Confirm → Enrich (TI/watchlist) → Create Rule → Automate (playbook)')


## SC-200 hunting exam tips

### Sentinel-specific hunting features

| Feature | What it does |
|---------|-------------|
| **Hunting queries** | Pre-built KQL queries organized by MITRE tactic |
| **Bookmarks** | Save interesting query results for later investigation |
| **Livestream** | Real-time query results as events flow in |
| **Notebooks** | Jupyter notebooks connected to Sentinel data |
| **Hunting graph** | Visual entity relationships |
| **Watchlists** | Shared lookup tables for IOCs / allow-deny lists |
| **Data lake tier** | Low-cost storage for long-term hunting data |

### Advanced Hunting in Defender XDR

| Feature | What it does |
|---------|-------------|
| **Custom detection rules** | Save a hunting query as a scheduled detection |
| **Shared queries** | Share with team |
| **Blast radius** | Graph showing how far an attack could spread |
| **Go hunt** | One-click hunting from an incident or entity |

### The hunting cycle

```
    ┌──────────────┐
    │ Threat Intel │ ← external IOCs, MITRE techniques
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Hypothesis  │ ← "attacker may be doing X"
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Query Data  │ ← KQL across relevant tables
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Enrich      │ ← join watchlists / TI
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Analyze     │ ← confirm / refute
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Action      │ ← create rule + playbook, or refine hypothesis
    └──────┬───────┘
           │
           └──────────▶ repeat
```

**Next**: [Notebook 3 — Baselines, Anomalies & the Hunting Maturity Model](03_baselines_and_anomalies.ipynb)
